# Comorbidity Network Analysis Pipeline

This notebook reproduces the full analysis pipeline described in:

> *"Comorbidity networks via sequence embedding and Gaussian graphical models"*

**Pipeline steps:**
1. Data import, preprocessing, and sequence visualization
2. Word embedding (BioWordVec fine-tuning)
3. Cosine-similarity matrices
4. Comparison, risk-factor analysis, and sensitivity analysis
5. Network sparsification via Graphical Lasso (GGM)
6. GGM network visualization
7. Topological analysis and community detection

> **NOTE (Simulated data):** When `DATA_MODE = "simulated"`, the pipeline
> runs on synthetic data generated by `simulate_sequences.py`. Simulated
> patients do not represent real individuals. Network structures will differ
> from the original publication results.


## Section 0 — Configuration

Set `DATA_MODE = "simulated"` to use the synthetic sequences generated by
`simulate_sequences.py` (run it first: `python simulate_sequences.py`).
Set `DATA_MODE = "real"` and fill in the paths below to use your own data.
The real-data files must have columns: `ID`, `age_in_days_since_born`, `event_name`.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
from scipy.spatial.distance import cosine
from scipy.stats import mannwhitneyu, pearsonr, spearmanr
from scipy.special import erfc
from sklearn.covariance import GraphicalLassoCV
try:
    from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity
except Exception:
    # Fallback using scipy (works with older sklearn versions)
    from scipy.spatial.distance import cdist
    def sk_cosine_similarity(X):
        return 1 - cdist(X, X, metric="cosine")

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120})

# ── User configuration ────────────────────────────────────────────────────────
DATA_MODE = "simulated"   # "simulated" | "real"

# Paths for real data (only used when DATA_MODE = "real")
REAL_MAIN_SEQ_PATH     = None   # e.g. "/path/to/sequence_main_final.csv"
REAL_PARALLEL_SEQ_PATH = None   # e.g. "/path/to/sequence_parallel_final.csv"

# Output directories
OUTPUT_PLOTS_DIR = "plots"
OUTPUT_GGM_DIR   = "results_GGM"
OUTPUT_BASE_DIR  = "Baseline"
os.makedirs(OUTPUT_PLOTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_GGM_DIR,   exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR,  exist_ok=True)

print(f"DATA_MODE = {DATA_MODE}")
print("Configuration loaded.")


## Section 1 — Data Import and Preprocessing

### 1.1  Load sequences

Simulated sequences are produced by `simulate_sequences.py`.
Both files share the same column schema:

| Column | Description |
|---|---|
| `ID` | Patient identifier |
| `age_in_days_since_born` | Age at event (days) |
| `event_name` | MeSH disease / risk-factor name |


In [ ]:
if DATA_MODE == "simulated":
    MAIN_PATH     = os.path.join("data_simulated", "sequence_main_simulated.csv")
    PARALLEL_PATH = os.path.join("data_simulated", "sequence_parallel_simulated.csv")
else:
    MAIN_PATH     = REAL_MAIN_SEQ_PATH
    PARALLEL_PATH = REAL_PARALLEL_SEQ_PATH

df_main     = pd.read_csv(MAIN_PATH)
df_parallel = pd.read_csv(PARALLEL_PATH)

print(f"Main sequence:     {df_main.shape[0]:,} rows  |  {df_main['ID'].nunique():,} unique patients")
print(f"Parallel sequence: {df_parallel.shape[0]:,} rows  |  {df_parallel['ID'].nunique():,} unique patients")
df_main.head(3)


### 1.2  Filtering

* Remove events at age = 0 from the main sequence.
* Retain only patients with **≥ 2 distinct disease events** (mirrors the real-data cohort criterion).

In [ ]:
# Remove age-0 entries from main sequence
df_main = df_main[df_main["age_in_days_since_born"] > 0].copy()

# Keep patients with ≥ 2 distinct disease events
counts   = df_main.groupby("ID")["event_name"].nunique()
keep_ids = counts[counts >= 2].index
df_main     = df_main[df_main["ID"].isin(keep_ids)].copy()
df_parallel = df_parallel[df_parallel["ID"].isin(keep_ids)].copy()

print(f"After filtering:")
print(f"  Main sequence:     {len(df_main):,} rows  |  {df_main['ID'].nunique():,} patients")
print(f"  Parallel sequence: {len(df_parallel):,} rows  |  {df_parallel['ID'].nunique():,} patients")


### 1.3  MeSH vocabulary reference

The MeSH term sets used in the original analysis are defined below.
For simulated data the `event_name` column already contains MeSH terms;
for real data you would apply the mapping dictionaries from
`Comorb_network_seq_preprocessing.ipynb` (chunks 21 / 23) here.

In [ ]:
# ── MeSH names for the 24 main cardiometabolic diseases ──────────────────────
DISEASE_NAMES_MAIN = [
    "Hypertension", "Dyslipidemias", "Coronary Disease", "Diabetes Mellitus",
    "Atrial Fibrillation", "Acute Kidney Injury", "Cerebrovascular Disorders",
    "Kidney Diseases, Chronic, Stage 3", "Heart Failure",
    "Peripheral Vascular Diseases", "Pulmonary Embolism",
    "Mitral Valve Insufficiency", "Pacemaker, Artificial",
    "Stroke, Hemorrhagic", "Heart Arrest", "Tachycardia, Ventricular",
    "Kidney Diseases, Chronic, Stage 4", "Kidney Failure, Chronic, Endstage",
    "Cardiomyopathy, Dilated", "Kidney Diseases, Chronic, Stage 5",
    "Tricuspid Valve Insufficiency", "Stroke", "Heart Failure, Endstage",
    "Mitral Valve Stenosis",
]

# Keep only diseases that actually appear in the loaded data
DISEASE_NAMES_MAIN = [d for d in DISEASE_NAMES_MAIN
                      if d in df_main["event_name"].unique()]
print(f"Main diseases present in data: {len(DISEASE_NAMES_MAIN)}")

# ── MeSH names for parallel risk factors ─────────────────────────────────────
RF_NAMES = [n for n in df_parallel["event_name"].unique()
            if n not in DISEASE_NAMES_MAIN]
print(f"Risk factors present in data:  {len(RF_NAMES)}")

PHRASES_ALL = DISEASE_NAMES_MAIN + RF_NAMES   # full vocabulary
print(f"Total vocabulary size:         {len(PHRASES_ALL)}")


### 1.4  Build final sequence DataFrames

Save the processed sequences for downstream steps.

In [ ]:
# Main-only events (sorted by patient and age)
sequence_main_final = (
    df_main[df_main["event_name"].isin(DISEASE_NAMES_MAIN)]
    .sort_values(["ID", "age_in_days_since_born"])
    .reset_index(drop=True)
)

# Main + parallel (concatenate and sort)
df_parallel_filtered = df_parallel[df_parallel["event_name"].isin(RF_NAMES)].copy()
sequence_main_parr_final = (
    pd.concat([sequence_main_final, df_parallel_filtered])
    .sort_values(["ID", "age_in_days_since_born"])
    .reset_index(drop=True)
)

sequence_main_final.to_csv("sequence_main_final.csv", index=False)
sequence_main_parr_final.to_csv("sequence_main_parr_final.csv", index=False)

print(f"sequence_main_final:      {len(sequence_main_final):,} rows")
print(f"sequence_main_parr_final: {len(sequence_main_parr_final):,} rows")


### 1.5  Cohort summary statistics

In [ ]:
n_patients = sequence_main_final["ID"].nunique()
data_unique = sequence_main_final.groupby(["ID", "event_name"], as_index=False).first()

incidence_df = (
    data_unique.groupby("event_name")["ID"].nunique()
    .reset_index(name="n_patients_with_disease")
    .assign(incidence_pct=lambda x: x["n_patients_with_disease"] / n_patients * 100)
    .sort_values("incidence_pct", ascending=False)
    .reset_index(drop=True)
)
print(f"Total patients: {n_patients:,}")
print(f"Total main-sequence events: {len(sequence_main_final):,}")
display(incidence_df.style.format({"incidence_pct": "{:.1f}%"}))


### 1.6  Visualizations

Python equivalents of the plots produced by `plot_for_draft.R`.

In [ ]:
# ── Plot 1: Disease incidence by age (6×4 faceted) ──────────────────────────
# First incidence per patient per disease, convert to years
first_incidence = (
    sequence_main_final
    .sort_values(["ID", "age_in_days_since_born"])
    .groupby(["ID", "event_name"], as_index=False)
    .first()
)
first_incidence["age_yr"] = np.round(
    first_incidence["age_in_days_since_born"] / 365
).astype(int)

age_dist = (
    first_incidence
    .groupby(["event_name", "age_yr"])
    .size()
    .reset_index(name="N")
)
total_per_disease = age_dist.groupby("event_name")["N"].sum().reset_index(name="total")
age_dist = age_dist.merge(total_per_disease, on="event_name")
age_dist["pct"] = age_dist["N"] / age_dist["total"] * 100

diseases_sorted = incidence_df["event_name"].tolist()
n_diseases = len(diseases_sorted)
ncols, nrows = 4, int(np.ceil(n_diseases / 4))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 2.5), constrained_layout=True)
axes = axes.flatten()

for ax, disease in zip(axes, diseases_sorted):
    sub = age_dist[age_dist["event_name"] == disease]
    ax.scatter(sub["age_yr"], sub["pct"], color="steelblue", s=8, alpha=0.7)
    ax.set_title(disease, fontsize=7, wrap=True)
    ax.set_xlabel("Age (years)", fontsize=6)
    ax.set_ylabel("% of events", fontsize=6)
    ax.tick_params(labelsize=5)

for ax in axes[n_diseases:]:
    ax.set_visible(False)

fig.suptitle("Disease incidence by age", fontsize=12, y=1.01)
fig.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_incidence_by_age.png"),
            dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_incidence_by_age.svg"),
            format="svg", bbox_inches="tight")
plt.show()
print("Saved: disease_incidence_by_age.png / .svg")


In [ ]:
# ── Plot 2: Total disease events distribution by age ─────────────────────────
all_events_age = sequence_main_final.copy()
all_events_age["age_yr"] = np.round(
    all_events_age["age_in_days_since_born"] / 365
).astype(int)

total_by_age = all_events_age.groupby("age_yr").size().reset_index(name="N")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(total_by_age["age_yr"], total_by_age["N"], color="steelblue", alpha=0.8)
ax.set_xlabel("Age (Years)", fontsize=12)
ax.set_ylabel("Number of events", fontsize=12)
ax.set_title("Distribution of disease events by age (primary cardiometabolic diseases)",
             fontsize=12)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_PLOTS_DIR, "total_events_by_age.png"),
            dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_PLOTS_DIR, "total_events_by_age.svg"),
            format="svg", bbox_inches="tight")
plt.show()
print("Saved: total_events_by_age.png / .svg")


## Section 2 — Word Embedding

> **⚠️ USER DECISION POINT — Pre-trained model**
>
> This section requires the BioWordVec base model (`pubmed_mesh_test`).
> If you have not already trained it, run from a terminal **before** executing
> this section:
>
> ```bash
> python BioWordVec.py
> ```
>
> Training takes ~15–30 minutes depending on hardware. The model files
> (`pubmed_mesh_test`, `pubmed_mesh_test.bin`) will be saved in the current
> directory.


In [ ]:
from pathlib import Path
from gensim.models import FastText

PRETRAINED_MODEL_PATH  = "./pubmed_mesh_test"
FINETUNED_MAIN_PATH    = "./pubmed_mesh_finetuned_main"
FINETUNED_MAINPAR_PATH = "./pubmed_mesh_finetuned_main_par"

if not Path(PRETRAINED_MODEL_PATH).exists():
    raise FileNotFoundError(
        "Pre-trained model not found at './pubmed_mesh_test'.\n"
        "Run:  python BioWordVec.py\n"
        "then re-execute this cell."
    )
print("Pre-trained BioWordVec model found.")
base_model = FastText.load(PRETRAINED_MODEL_PATH)
print(f"  Vector size: {base_model.vector_size}  |  Vocab size: {len(base_model.wv)}")


> **⚠️ USER DECISION POINT — Fine-tuning hyperparameters**
>
> Adjust the parameters below before running the fine-tuning cells.
> Defaults mirror those used in the original analysis.

| Parameter | Recommended range | Effect |
|---|---|---|
| `FINETUNE_EPOCHS` | 3–10 | More epochs → better domain adaptation, longer runtime |
| `FINETUNE_WINDOW` | 3–7 | Larger → more distant co-occurrences captured |
| `FINETUNE_WORKERS` | 2–8 | Parallel threads (limit to available CPUs) |


In [ ]:
# ── USER DECISION POINT ───────────────────────────────────────────────────────
FINETUNE_EPOCHS  = 5    # recommended: 3–10
FINETUNE_WINDOW  = 5    # context window
FINETUNE_WORKERS = 4    # CPU threads
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
def build_corpus(seq_df: pd.DataFrame, phrase_col: str = "event_name") -> list:
    """Build a per-patient token corpus from a sequence DataFrame."""
    corpus = []
    for pid, grp in seq_df.sort_values("age_in_days_since_born").groupby("ID"):
        tokens = []
        for phrase in grp[phrase_col]:
            # Split multi-word MeSH terms into individual tokens
            tokens.extend(phrase.replace(",", "").lower().split())
        if tokens:
            corpus.append(tokens)
    return corpus

corpus_main    = build_corpus(sequence_main_final)
corpus_mainpar = build_corpus(sequence_main_parr_final)
print(f"Corpus (main):         {len(corpus_main):,} sentences")
print(f"Corpus (main+parallel): {len(corpus_mainpar):,} sentences")
print(f"Example sentence (main): {corpus_main[0][:10]} …")


In [ ]:
# ── Fine-tune on main sequence ────────────────────────────────────────────────
import copy
print("Fine-tuning on main sequence …")
model_main = copy.deepcopy(base_model)
model_main.build_vocab(corpus_main, update=True)
model_main.train(corpus_main,
                 total_examples=len(corpus_main),
                 epochs=FINETUNE_EPOCHS)
model_main.save(FINETUNED_MAIN_PATH)
print(f"  Saved → {FINETUNED_MAIN_PATH}")


In [ ]:
# ── Fine-tune on main + parallel sequence ─────────────────────────────────────
print("Fine-tuning on main + parallel sequence …")
model_mainpar = copy.deepcopy(base_model)
model_mainpar.build_vocab(corpus_mainpar, update=True)
model_mainpar.train(corpus_mainpar,
                    total_examples=len(corpus_mainpar),
                    epochs=FINETUNE_EPOCHS)
model_mainpar.save(FINETUNED_MAINPAR_PATH)
print(f"  Saved → {FINETUNED_MAINPAR_PATH}")


### 2.1  Extract phrase embeddings and build patient-level embedding matrix

For each disease / risk-factor phrase, we average its word-level embeddings to obtain a
phrase-level vector.  Then, for each patient we record the phrase vectors for all events
they experienced (averaged across repeat events).  Finally, we pivot to a wide matrix
`(patients × phrases)` used as input to the GGM estimation.

In [ ]:
def phrase_vector(phrase: str, model) -> np.ndarray:
    """Average embedding for a multi-word MeSH phrase (handles OOV via FastText subwords)."""
    tokens = phrase.replace(",", "").lower().split()
    vecs   = [model.wv.get_vector(t, norm=True) for t in tokens]
    return np.mean(vecs, axis=0)

def build_phrase_embedding_matrix(seq_df: pd.DataFrame,
                                  phrases: list,
                                  model) -> pd.DataFrame:
    """
    Build a (n_patients × n_phrases) embedding matrix.
    Each cell = mean phrase vector across all events of that type for that patient.
    Missing (patient, phrase) pairs are filled with the global phrase vector.
    """
    global_vecs = {p: phrase_vector(p, model) for p in phrases}
    records = {}
    for pid, grp in seq_df.groupby("ID"):
        row = {}
        for phrase in phrases:
            sub = grp[grp["event_name"] == phrase]
            if len(sub) > 0:
                vecs = np.array([phrase_vector(phrase, model)] * len(sub))
                row[phrase] = vecs.mean(axis=0)
            else:
                row[phrase] = global_vecs[phrase]
        records[pid] = np.concatenate([row[p] for p in phrases])
    # Column names: phrase_dim0, phrase_dim1, … (flat)
    # For GGM we want one column per phrase (averaged across dims)
    # Use scalar: mean of absolute embedding components as feature
    rows = []
    for pid, grp in seq_df.groupby("ID"):
        row = {"ID": pid}
        for phrase in phrases:
            sub = grp[grp["event_name"] == phrase]
            if len(sub) > 0:
                v = phrase_vector(phrase, model)
            else:
                v = global_vecs[phrase]
            row[phrase] = float(np.mean(np.abs(v)))
        rows.append(row)
    return pd.DataFrame(rows).set_index("ID")

print("Building patient-level embedding matrix (main + parallel) …")
phrase_emb_df = build_phrase_embedding_matrix(
    sequence_main_parr_final, PHRASES_ALL, model_mainpar
)
phrase_emb_df.to_csv("phrase_embeddings_main_par_df.csv")
print(f"Embedding matrix shape: {phrase_emb_df.shape}  (patients × phrases)")
phrase_emb_df.head(3)


## Section 3 — Similarity Construction and Visualization

In [ ]:
def get_phrase_vectors(phrases: list, model) -> np.ndarray:
    """Return (n_phrases × vector_size) array of phrase embeddings."""
    return np.array([phrase_vector(p, model) for p in phrases])

# ── 24×24 cosine similarity (main model) ────────────────────────────────────
vecs_main    = get_phrase_vectors(DISEASE_NAMES_MAIN, model_main)
sim_main_arr = sk_cosine_similarity(vecs_main)
sim_main_df  = pd.DataFrame(sim_main_arr,
                             index=DISEASE_NAMES_MAIN,
                             columns=DISEASE_NAMES_MAIN)

# ── Full 121×121 cosine similarity (main+parallel model) ────────────────────
vecs_all         = get_phrase_vectors(PHRASES_ALL, model_mainpar)
sim_all_arr      = sk_cosine_similarity(vecs_all)
sim_all_df       = pd.DataFrame(sim_all_arr, index=PHRASES_ALL, columns=PHRASES_ALL)

# ── 24×24 submatrix from the joint model (for comparison) ───────────────────
sim_mainpar_24_df = sim_all_df.loc[DISEASE_NAMES_MAIN, DISEASE_NAMES_MAIN]

# ── Global vmin / vmax across both 24×24 matrices (consistent colour scale) ──
_vals  = np.concatenate([sim_main_arr.flatten(),
                          sim_mainpar_24_df.values.flatten()])
VMIN_SIM = float(np.min(_vals))
VMAX_SIM = float(np.max(_vals))
print(f"Similarity range (both models): [{VMIN_SIM:.3f}, {VMAX_SIM:.3f}]")


In [ ]:
def plot_sim_heatmap(df: pd.DataFrame, title: str, fname: str,
                     vmin: float = None, vmax: float = None,
                     cmap: str = "Blues") -> None:
    """Lower-triangle seaborn heatmap for a square similarity matrix."""
    mask = np.triu(np.ones(df.shape, dtype=bool), k=0)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(df, mask=mask, cmap=cmap, annot=False, square=True,
                linewidths=0.3, cbar_kws={"shrink": 0.8},
                vmin=vmin, vmax=vmax, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, fname), format="svg", bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, fname.replace(".svg", ".pdf")),
                bbox_inches="tight")
    plt.show()

plot_sim_heatmap(sim_main_df,
                 "Disease Similarity Heatmap — Main Sequence Model",
                 "disease_similarity_heatmap_main.svg",
                 vmin=VMIN_SIM, vmax=VMAX_SIM)


In [ ]:
plot_sim_heatmap(sim_mainpar_24_df,
                 "Disease Similarity Heatmap — Main + Parallel Model",
                 "disease_similarity_heatmap_main_par.svg",
                 vmin=VMIN_SIM, vmax=VMAX_SIM)


In [ ]:
# ── Helper functions for network visualisation (ported from GGM_plot.ipynb) ──

def weighted_graph_construction(correlation_df: pd.DataFrame,
                                  threshold: float = 0.0) -> nx.Graph:
    """Build undirected weighted NetworkX graph from a correlation/similarity DataFrame."""
    corr_arr   = np.abs(np.array(correlation_df))
    G          = nx.DiGraph()
    prot_names = list(correlation_df.columns)
    G.add_nodes_from(prot_names)
    n = corr_arr.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            if corr_arr[i, j] > threshold:
                G.add_edge(prot_names[i], prot_names[j], weight=corr_arr[i, j])
    print(f"  #edges / n²: {len(G.edges()) / n**2:.4f}")
    return G.to_undirected()


def plot_weighted_graph_colored(G: nx.Graph, title: str = "",
                                 cmap_name: str = "coolwarm",
                                 layout: str = "circular",
                                 fname: str = None) -> None:
    """Plot weighted graph with edge colours proportional to weight."""
    fig, ax = plt.subplots(figsize=(14, 10))
    if layout == "spring":
        pos = nx.spring_layout(G, seed=1)
    elif layout == "kamada_kawai":
        pos = nx.kamada_kawai_layout(G)
    else:
        pos = nx.circular_layout(G)

    if G.number_of_edges() == 0:
        nx.draw_networkx(G, pos, ax=ax, node_color="lightgray",
                         node_size=600, font_size=9)
        ax.set_title(title or "Graph (no edges)")
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        return

    weights  = [G[u][v].get("weight", 1.0) for u, v in G.edges()]
    min_w, max_w = min(weights), max(weights)
    cmap     = plt.colormaps[cmap_name]
    norm     = mcolors.Normalize(vmin=min_w if min_w < max_w else 0,
                                  vmax=max_w if min_w < max_w else 1)
    edge_colors = []
    for w in weights:
        base = cmap(norm(w))
        alpha = 0.1 + 0.9 * norm(w)
        edge_colors.append((*base[:3], alpha))

    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=600, node_color="lightgray")
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color="black")
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors, width=2)

    sm = cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    plt.colorbar(sm, ax=ax, shrink=0.7).set_label("Similarity", fontsize=11)
    ax.set_title(title or "Graph Colored by Edge Similarity", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    if fname:
        plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, fname),
                    format="svg", bbox_inches="tight")
    plt.show()


# ── Network graph — main similarity ──────────────────────────────────────────
G_main = weighted_graph_construction(sim_main_df, threshold=0.0)
plot_weighted_graph_colored(G_main, title="Disease Similarity Network — Main Model",
                             cmap_name="Blues", layout="circular",
                             fname="graph_similarity_plot_main.svg")


In [ ]:
# ── Network graph — main+parallel similarity ──────────────────────────────────
G_mainpar = weighted_graph_construction(sim_mainpar_24_df, threshold=0.0)
plot_weighted_graph_colored(G_mainpar,
                             title="Disease Similarity Network — Main+Parallel Model",
                             cmap_name="Blues", layout="circular",
                             fname="graph_similarity_plot_main_par.svg")


## Section 4 — Comparison and Risk Factor Analysis

In [ ]:
# ── Signed and absolute difference matrices ───────────────────────────────────
sim_diff_signed = sim_mainpar_24_df.subtract(sim_main_df)  # main_par - main
sim_diff_abs    = sim_diff_signed.abs()

mask_tri = np.triu(np.ones(sim_diff_signed.shape, dtype=bool), k=0)
fig, ax  = plt.subplots(figsize=(13, 10))
vmax_d   = float(np.nanmax(np.abs(sim_diff_signed.values)))
sns.heatmap(sim_diff_signed, mask=mask_tri, cmap="RdBu_r", center=0,
            vmin=-vmax_d, vmax=vmax_d, annot=False, square=True,
            linewidths=0.1, cbar_kws={"shrink": 0.8,
                                       "label": "Change in Similarity\n(+ = Increased)"},
            ax=ax)
ax.set_title("Signed Changes in Disease Similarity\n(Red = Increased, Blue = Decreased)",
             fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_similarity_changes_signed.svg"),
            format="svg", bbox_inches="tight")
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_similarity_changes_signed.pdf"),
            bbox_inches="tight")
plt.show()


### 4.1  User decision point — risk-factor threshold and top-N pairs

> **⚠️ USER DECISION POINT**
>
> * `RF_CORRELATION_THRESHOLD`: minimum cosine similarity between a risk factor and a disease
>   to consider the risk factor "highly correlated" with that disease.
>   **Default = median of the risk-factor × risk-factor similarity submatrix** (mirrors
>   the criterion used in the original analysis).
> * `TOP_N_PAIRS`: number of most-affected disease pairs to highlight in the analysis.
>   Sensitivity analysis below lets you explore multiple thresholds automatically.


In [ ]:
# ── Risk-factor × risk-factor similarity submatrix ────────────────────────────
sim_rf_rf = sim_all_df.loc[RF_NAMES, RF_NAMES].values

# ── USER DECISION POINT ───────────────────────────────────────────────────────
RF_CORRELATION_THRESHOLD = float(np.median(sim_rf_rf))  # ← override if desired
TOP_N_PAIRS              = 14   # ← number of top affected disease pairs
# ─────────────────────────────────────────────────────────────────────────────

print(f"RF correlation threshold (median): {RF_CORRELATION_THRESHOLD:.4f}")
print(f"Top N pairs to analyse:            {TOP_N_PAIRS}")


In [ ]:
# ── Identify top affected disease pairs ───────────────────────────────────────
upper_idx = np.triu_indices_from(sim_diff_signed.values, k=1)
pairs_list = []
for i, j in zip(*upper_idx):
    d1, d2 = DISEASE_NAMES_MAIN[i], DISEASE_NAMES_MAIN[j]
    change  = float(sim_diff_signed.iloc[i, j])
    pairs_list.append({"disease1": d1, "disease2": d2,
                        "change": change,
                        "abs_change": abs(change),
                        "direction": "increased" if change > 0 else "decreased"})

pairs_df = pd.DataFrame(pairs_list).sort_values("abs_change", ascending=False).reset_index(drop=True)
top_affected_pairs = pairs_df.head(TOP_N_PAIRS).to_dict("records")

print(f"Top {TOP_N_PAIRS} most affected disease pairs:")
display(pairs_df.head(TOP_N_PAIRS)[["disease1", "disease2", "change", "direction"]]
        .style.format({"change": "{:+.4f}"}))


In [ ]:
# ── Risk-factor correlation functions ─────────────────────────────────────────

def find_correlated_risk_factors(disease_name: str,
                                  risk_factors: list,
                                  similarity_matrix: np.ndarray,
                                  all_phrases: list,
                                  threshold: float = 0.5,
                                  top_n: int = 10) -> list:
    """Find risk factors with cosine similarity ≥ threshold to a given disease."""
    if disease_name not in all_phrases:
        return []
    d_idx = all_phrases.index(disease_name)
    corrs = []
    for rf in risk_factors:
        if rf in all_phrases:
            rf_idx = all_phrases.index(rf)
            corr   = float(similarity_matrix[d_idx, rf_idx])
            if corr >= threshold:
                corrs.append({"risk_factor": rf, "correlation": corr})
    return sorted(corrs, key=lambda x: -x["correlation"])[:top_n]


def analyze_pair_shared_risk_factors(pair: dict,
                                      disease_correlations: dict) -> dict:
    """Compute shared high-correlated risk factors for a disease pair."""
    d1, d2 = pair["disease1"], pair["disease2"]
    rf1 = {c["risk_factor"] for c in disease_correlations.get(d1, [])}
    rf2 = {c["risk_factor"] for c in disease_correlations.get(d2, [])}
    shared = rf1 & rf2
    n_max  = max(len(rf1), len(rf2), 1)
    return {
        "pair":             f"{d1} ↔ {d2}",
        "change":           pair["change"],
        "n_rf_d1":          len(rf1),
        "n_rf_d2":          len(rf2),
        "n_shared":         len(shared),
        "normalized_shared": len(shared) / n_max,
    }


def analyze_disease_risk_correlations(diseases: list,
                                       risk_factors: list,
                                       all_phrases: list,
                                       similarity_matrix: np.ndarray,
                                       threshold: float = 0.5) -> dict:
    """Return {disease: [correlated risk factors]} for all diseases."""
    return {
        d: find_correlated_risk_factors(
            d, risk_factors, similarity_matrix, all_phrases, threshold
        )
        for d in diseases
    }


In [ ]:
# ── Risk-factor analysis at the selected threshold ────────────────────────────
disease_correlations = analyze_disease_risk_correlations(
    DISEASE_NAMES_MAIN, RF_NAMES, PHRASES_ALL,
    sim_all_arr, threshold=RF_CORRELATION_THRESHOLD
)

pair_analyses = [
    analyze_pair_shared_risk_factors(p, disease_correlations)
    for p in top_affected_pairs
]
df_pairs = pd.DataFrame(pair_analyses)
print("Shared risk-factor analysis for top affected pairs:")
display(df_pairs.style.format({
    "change": "{:+.4f}",
    "normalized_shared": "{:.3f}"
}))


### 4.2  Sensitivity analysis

Iterate over multiple threshold values and track how the shared risk-factor results
and the Mann-Whitney U test change.

> **⚠️ USER DECISION POINT** — edit `THRESHOLD_VALUES` to explore other values.

In [ ]:
# ── USER DECISION POINT ───────────────────────────────────────────────────────
THRESHOLD_VALUES = [0.3, 0.5, 0.7, 0.9]   # ← add or remove values as needed
# ─────────────────────────────────────────────────────────────────────────────

all_sensitivity_results = []

for t in THRESHOLD_VALUES:
    dc_t = analyze_disease_risk_correlations(
        DISEASE_NAMES_MAIN, RF_NAMES, PHRASES_ALL, sim_all_arr, threshold=t
    )
    pa_t = [analyze_pair_shared_risk_factors(p, dc_t) for p in top_affected_pairs]
    df_t = pd.DataFrame(pa_t)

    # Mann-Whitney U: top-N pairs vs remaining pairs
    rest_pairs = pairs_df.iloc[TOP_N_PAIRS:].to_dict("records")
    dc_rest    = analyze_disease_risk_correlations(
        DISEASE_NAMES_MAIN, RF_NAMES, PHRASES_ALL, sim_all_arr, threshold=t
    )
    pa_rest = [analyze_pair_shared_risk_factors(p, dc_rest) for p in rest_pairs]
    df_rest = pd.DataFrame(pa_rest) if pa_rest else pd.DataFrame(
        columns=df_t.columns)

    group_top  = df_t["normalized_shared"].dropna().values
    group_rest = df_rest["normalized_shared"].dropna().values if len(df_rest) > 0 else np.array([0.0])

    if len(group_top) > 0 and len(group_rest) > 0:
        stat, pval = mannwhitneyu(group_top, group_rest, alternative="greater")
    else:
        stat, pval = np.nan, np.nan

    # Pearson / Spearman between |change| and n_shared
    changes = df_t["change"].abs().values
    shared  = df_t["n_shared"].values
    if len(changes) > 2:
        pr, pp  = pearsonr(changes, shared)
        sr, sp  = spearmanr(changes, shared)
    else:
        pr = pp = sr = sp = np.nan

    all_sensitivity_results.append({
        "threshold":         t,
        "mw_stat":           stat,
        "mw_pval":           pval,
        "pearson_r":         pr,
        "pearson_p":         pp,
        "spearman_r":        sr,
        "spearman_p":        sp,
        "mean_shared_top":   group_top.mean() if len(group_top) else np.nan,
        "mean_shared_rest":  group_rest.mean() if len(group_rest) else np.nan,
    })
    print(f"Threshold {t:.2f}: MWU p={pval:.4g}  Pearson r={pr:.3f} p={pp:.4g}")

df_sensitivity = pd.DataFrame(all_sensitivity_results)
print("\nSensitivity analysis summary:")
display(df_sensitivity.style.format({
    c: "{:.4f}" for c in df_sensitivity.select_dtypes(float).columns
}))


In [ ]:
# ── Fisher Z-difference test ──────────────────────────────────────────────────
def fisher_z_diff_test(R1: np.ndarray, R2: np.ndarray,
                        n: int, alpha: float = 0.05,
                        clip_eps: float = 1e-7):
    """
    Two-sided Fisher Z-transform test for differences between two correlation matrices.
    Bonferroni correction: alpha / (p*(p-1)/2).
    """
    R1 = np.clip(np.asarray(R1, float), -1 + clip_eps, 1 - clip_eps)
    R2 = np.clip(np.asarray(R2, float), -1 + clip_eps, 1 - clip_eps)
    z1, z2 = np.arctanh(R1), np.arctanh(R2)
    se     = np.sqrt(2.0 / max(n - 3, 1))
    Z_diff = (z1 - z2) / se
    np.fill_diagonal(Z_diff, 0.0)
    pvals  = erfc(np.abs(Z_diff) / np.sqrt(2.0))
    np.fill_diagonal(pvals, 1.0)
    p = R1.shape[0]
    n_tests    = p * (p - 1) // 2
    alpha_corr = alpha / n_tests if n_tests > 0 else alpha
    sig = pvals < alpha_corr
    np.fill_diagonal(sig, False)
    return Z_diff, pvals, sig

n_patients_est = sequence_main_final["ID"].nunique()
Z_diff, pvals_fisher, sig_fisher = fisher_z_diff_test(
    sim_main_df.values, sim_mainpar_24_df.values, n=n_patients_est
)
n_significant = sig_fisher.sum() // 2  # upper triangle only
print(f"Fisher Z-test: {n_significant} significant pair-wise changes")
print(f"  (Bonferroni α = {0.05/max(len(DISEASE_NAMES_MAIN)*(len(DISEASE_NAMES_MAIN)-1)//2,1):.4g})")


## Section 5 — Network Sparsification (Graphical Lasso / GGM)

We estimate a sparse inverse covariance (precision) matrix from the patient-level
phrase-embedding matrix using cross-validated Graphical Lasso (scikit-learn).
The selected regularisation parameter `alpha_` is printed for transparency.

In [ ]:
emb_df = pd.read_csv("phrase_embeddings_main_par_df.csv", index_col=0)
print(f"Embedding matrix: {emb_df.shape}  (patients × phrases)")

# Standardise columns before fitting
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(emb_df.values)

print("Fitting GraphicalLassoCV (cross-validated Graphical Lasso) …")
glasso = GraphicalLassoCV(cv=5, n_jobs=-1, max_iter=500)
glasso.fit(X)

precision_mat = glasso.precision_
print(f"  Selected alpha (lambda): {glasso.alpha_:.6f}")
print(f"  Precision matrix shape:  {precision_mat.shape}")

adj_GGM_df = pd.DataFrame(precision_mat,
                            index=emb_df.columns,
                            columns=emb_df.columns)
adj_GGM_df.to_csv(os.path.join(OUTPUT_GGM_DIR, "GGM_adj_main_par_full.csv"))
print(f"  Saved → {os.path.join(OUTPUT_GGM_DIR, 'GGM_adj_main_par_full.csv')}")


## Section 6 — GGM Network Visualisation

In [ ]:
# Load GGM results (converges all paths)
adj_GGM_df = pd.read_csv(
    os.path.join(OUTPUT_GGM_DIR, "GGM_adj_main_par_full.csv"), index_col=0
)

# Extract 24×24 disease-only submatrix
common_diseases = [d for d in DISEASE_NAMES_MAIN
                   if d in adj_GGM_df.index and d in adj_GGM_df.columns]
adj_GGM_df_main = adj_GGM_df.loc[common_diseases, common_diseases]
print(f"GGM matrix (diseases only): {adj_GGM_df_main.shape}")


In [ ]:
# ── Weighted heatmap (coolwarm, lower-triangle) ───────────────────────────────
mask = np.triu(np.ones(adj_GGM_df_main.shape, dtype=bool), k=0)
vmax_g = float(np.nanmax(np.abs(adj_GGM_df_main.values)))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(adj_GGM_df_main, mask=mask, cmap="coolwarm", center=0,
            vmin=-vmax_g, vmax=vmax_g, annot=False, square=True,
            linewidths=0.3, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("GGM Precision Matrix — Disease Network", fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_GGM_heatmap_main_par_weighted.svg"),
            format="svg", bbox_inches="tight")
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_GGM_heatmap_main_par_weighted.pdf"),
            bbox_inches="tight")
plt.show()


In [ ]:
# ── Binary heatmap (edge presence / absence) ──────────────────────────────────
from matplotlib.colors import ListedColormap
adj_binary = (adj_GGM_df_main.abs() > 0).astype(float)
mask = np.triu(np.ones(adj_binary.shape, dtype=bool), k=0)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(adj_binary, mask=mask,
            cmap=ListedColormap(["lightgray", "#1E4586"]),
            annot=False, square=True, linewidths=0.3,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("GGM Disease Network — Binary Edges", fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_GGM_heatmap_main_par_binary.svg"),
            format="svg", bbox_inches="tight")
plt.show()


In [ ]:
# ── Normalised heatmap with X on zero edges ────────────────────────────────────
adj_GGM_abs = adj_GGM_df_main.abs()
_max_val = float(adj_GGM_abs.max().max())
adj_GGM_norm = adj_GGM_abs / _max_val if _max_val > 0 else adj_GGM_abs

mask = np.triu(np.ones(adj_GGM_norm.shape, dtype=bool), k=0)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(adj_GGM_norm, mask=mask, cmap="Blues", vmin=0, vmax=1,
            annot=False, square=True, linewidths=0.3,
            cbar_kws={"shrink": 0.8}, ax=ax)

# Mark zero-weight edges with an X
zero_mask  = (adj_GGM_df_main.values == 0)
lower_mask = np.tril(np.ones_like(zero_mask, bool), k=-1)
mark       = zero_mask & lower_mask
ys, xs     = np.where(mark)
ax.scatter(xs + 0.5, ys + 0.5, marker="x", s=30, linewidths=1.0, color="black")

ax.set_title("GGM Disease Network — Normalised Weights (X = absent edge)", fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_GGM_heatmap_main_par_weighted_normalized.svg"),
            format="svg", bbox_inches="tight")
plt.show()


In [ ]:
# ── Weighted network graph (circular layout) ───────────────────────────────────
G_ggm = weighted_graph_construction(adj_GGM_df_main, threshold=0.0)
plot_weighted_graph_colored(G_ggm,
                             title="GGM Disease Network (weighted)",
                             cmap_name="coolwarm", layout="circular",
                             fname="graph_GGM_plot_main_weighted.svg")


In [ ]:
# ── Full 121×121 GGM heatmap ─────────────────────────────────────────────────
mask = np.triu(np.ones(adj_GGM_df.shape, dtype=bool), k=0)
vmax_full = float(np.nanmax(np.abs(adj_GGM_df.values)))

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(adj_GGM_df, mask=mask, cmap="coolwarm", center=0,
            vmin=-vmax_full, vmax=vmax_full, annot=False, square=True,
            linewidths=0.1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("GGM Full Precision Matrix (diseases + risk factors)", fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "disease_GGM_heatmap_main_par_full_weighted.svg"),
            format="svg", bbox_inches="tight")
plt.show()


## Section 7 — Topological Analysis and Community Detection

Python port of `Network_description_comorb_network_old.Rmd`.

In [ ]:
from networkx.algorithms.community import (
    greedy_modularity_communities,
    girvan_newman,
)

# Build NetworkX graph from GGM adjacency (absolute values, no self-loops)
adj_abs = adj_GGM_df_main.abs()
G_topo  = nx.from_pandas_adjacency(adj_abs)
G_topo.remove_edges_from(list(nx.selfloop_edges(G_topo)))

# Remove zero-weight edges
G_topo = nx.Graph([
    (u, v, d) for u, v, d in G_topo.edges(data=True) if d.get("weight", 0) > 0
])
print(f"Network:  {G_topo.number_of_nodes()} nodes  |  {G_topo.number_of_edges()} edges")


In [ ]:
# ── Basic network statistics ──────────────────────────────────────────────────
edge_density = nx.density(G_topo)
try:
    diameter = nx.diameter(G_topo)
except nx.NetworkXError:
    diameter = "disconnected"

net_stats = pd.DataFrame({
    "Metric": ["Nodes", "Edges", "Edge Density", "Diameter"],
    "Value":  [G_topo.number_of_nodes(), G_topo.number_of_edges(),
               round(edge_density, 4), diameter],
})
print("Basic network statistics:")
display(net_stats.set_index("Metric"))


In [ ]:
# ── Edge weight distribution ──────────────────────────────────────────────────
edge_weights = [d["weight"] for _, _, d in G_topo.edges(data=True)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(edge_weights, bins=40, color="#1E4586", edgecolor="white")
axes[0].set_xlabel("Edge weight"); axes[0].set_ylabel("Count")
axes[0].set_title("Edge Weight Distribution")

from scipy.stats import gaussian_kde
kde = gaussian_kde(edge_weights)
xs  = np.linspace(min(edge_weights), max(edge_weights), 200)
axes[1].plot(xs, kde(xs), color="#1E4586", linewidth=2)
axes[1].fill_between(xs, kde(xs), alpha=0.3, color="#1E4586")
axes[1].set_xlabel("Edge weight"); axes[1].set_ylabel("Density")
axes[1].set_title("Edge Weight Density")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, "edge_weight_distribution.svg"),
            format="svg", bbox_inches="tight")
plt.show()


In [ ]:
# ── Community detection ───────────────────────────────────────────────────────

# 1. Fast Greedy (greedy modularity — NetworkX built-in; equivalent to R cluster_fast_greedy)
cfg_communities = list(greedy_modularity_communities(G_topo, weight=None))
cfg_map         = {}
for comm_id, comm in enumerate(cfg_communities):
    for node in comm:
        cfg_map[node] = comm_id
print(f"Fast Greedy: {len(cfg_communities)} communities")

# 2. Leiden algorithm (requires leidenalg + igraph)
leiden_map = None
try:
    import igraph as ig
    import leidenalg
    G_ig     = ig.Graph.from_networkx(G_topo)
    partition = leidenalg.find_partition(G_ig, leidenalg.ModularityVertexPartition)
    leiden_map = {G_ig.vs[i]["_nx_name"]: m
                  for i, m in enumerate(partition.membership)}
    print(f"Leiden:      {max(leiden_map.values())+1} communities")
except ImportError:
    print("leidenalg / igraph not installed — skipping Leiden (install via: pip install leidenalg igraph)")

# 3. Edge Betweenness (Girvan-Newman; primary assignment to match R Rmd)
gn_gen      = girvan_newman(G_topo)
gn_first    = next(gn_gen)
btw_map     = {}
for comm_id, comm in enumerate(gn_first):
    for node in comm:
        btw_map[node] = comm_id
print(f"Edge Betweenness: {len(gn_first)} communities")


In [ ]:
# ── Node centrality ───────────────────────────────────────────────────────────
degree_cent     = nx.degree_centrality(G_topo)
betweenness_cent = nx.betweenness_centrality(G_topo, weight=None)

topology_df = pd.DataFrame({
    "Node":                [n for n in G_topo.nodes()],
    "cfg_community":       [cfg_map.get(n, -1) for n in G_topo.nodes()],
    "btw_community":       [btw_map.get(n, -1) for n in G_topo.nodes()],
    "leiden_community":    [leiden_map.get(n, -1) if leiden_map else -1
                            for n in G_topo.nodes()],
    "centrality_degree":   [G_topo.degree(n) for n in G_topo.nodes()],
    "betweenness":         [betweenness_cent[n] for n in G_topo.nodes()],
})
topology_df["centrality_degree_rank"] = (
    topology_df["centrality_degree"].rank(ascending=False, method="min").astype(int)
)
topology_df["centrality_betweenness_rank"] = (
    topology_df["betweenness"].rank(ascending=False, method="min").astype(int)
)
topology_df = topology_df.sort_values("centrality_degree_rank").reset_index(drop=True)

out_csv = os.path.join(OUTPUT_BASE_DIR, "Node_Community_detection_centrality.csv")
topology_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
display(topology_df.style.format({"betweenness": "{:.4f}"}))


### 7.1  Community-coloured network visualisation

Nodes are coloured by community (Fast Greedy assignment by default).

In [ ]:
def plot_graph_by_community(G: nx.Graph,
                             community_df: pd.DataFrame,
                             layout: str = "kamada_kawai",
                             node_size: int = 650,
                             label_offset: float = 0.04,
                             edge_color: str = "lightgray",
                             edge_width: float = 1.8,
                             out: str = "graph_GGM_by_community.svg",
                             separate_communities: bool = True,
                             separation_strength: float = 1.8) -> None:
    """Community-coloured network plot (ported from Comorb_network_GGM_plot.ipynb)."""
    fig, ax = plt.subplots(figsize=(16, 12))

    if layout == "circular":
        pos = nx.circular_layout(G)
    elif layout == "spring":
        pos = nx.spring_layout(G, seed=1, k=0.9)
    else:
        pos = nx.kamada_kawai_layout(G)

    node_to_comm = dict(zip(community_df["Node"], community_df["cfg_community"]))

    if separate_communities:
        comm_nodes = {}
        for n in G.nodes():
            c = node_to_comm.get(n, -1)
            comm_nodes.setdefault(c, []).append(n)
        comm_centroid = {c: np.mean([pos[n] for n in ns], axis=0)
                        for c, ns in comm_nodes.items()}
        global_center = np.mean(list(pos.values()), axis=0)
        for c, nodes in comm_nodes.items():
            v    = comm_centroid[c] - global_center
            norm = np.linalg.norm(v)
            v    = v / norm if norm > 0 else np.array([1.0, 0.0])
            for n in nodes:
                pos[n] = pos[n] + separation_strength * v

    comms         = sorted({node_to_comm.get(n, -1) for n in G.nodes()})
    cmap_tab      = plt.colormaps["tab20"]
    comm_to_color = {c: cmap_tab(i / max(1, len(comms) - 1))
                     for i, c in enumerate(comms)}
    node_colors   = [comm_to_color[node_to_comm.get(n, -1)] for n in G.nodes()]

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_color,
                            width=edge_width, alpha=0.8)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                            node_size=node_size, linewidths=0.8,
                            edgecolors="white")
    center    = np.mean(list(pos.values()), axis=0)
    pos_labels = {}
    for n, (x, y) in pos.items():
        v  = np.array([x, y]) - center
        nn = np.linalg.norm(v)
        v  = v / nn if nn > 0 else np.array([1.0, 0.0])
        pos_labels[n] = (x + label_offset * v[0], y + label_offset * v[1])
    nx.draw_networkx_labels(G, pos_labels, ax=ax, font_size=9, font_color="black")

    ax.set_title("GGM Disease Network — Community Detection", fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, out), format="svg", bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_PLOTS_DIR, out.replace(".svg", ".pdf")),
                bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")


plot_graph_by_community(
    G_topo, topology_df,
    layout="kamada_kawai",
    separate_communities=True,
    separation_strength=1.8,
    out="graph_GGM_by_community.svg",
)


---

### Pipeline complete ✓

All outputs are saved in:
* `plots/` — all figures (SVG + PDF)
* `results_GGM/` — GGM precision matrix CSV
* `Baseline/` — node centrality and community CSV
* `sequence_main_final.csv` and `sequence_main_parr_final.csv`
